In [8]:

from pathlib import Path
import numpy as np
import pandas as pd
import lightgbm as lgb
from catboost import CatBoostRegressor
from scipy.stats import ks_2samp
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score, r2_score

RANDOM_STATE = 42
N_FOLDS = 4
N_JOBS = 4
INSPECTION_COST = 10.0
SHORTAGE_RATE = 0.20
SURPLUS_RATE = 0.02
BUDGET_PROB_SCALE = 1.30
INSPECTION_RISK_SCALE = 1.50
TWEEDIE_RISK_WEIGHT = 0.25




In [9]:
def locate_data():
    candidates = [Path('/kaggle/input'), Path.cwd(), Path('/mnt/data')]
    needed = {'train.csv', 'test.csv', 'sample_submission.csv'}
    for root in candidates:
        if not root.is_dir():
            continue
        for folder in ([root] + sorted({p.parent for p in root.rglob('train.csv')}) if root == Path('/kaggle/input') else [root]):
            if all((folder / f).is_file() for f in needed):
                return folder
    raise FileNotFoundError('Missing competition CSV files. Attach ML Project 2 using Kaggle Add Input.')


def preprocess(train, test):
    """Build raw and per-depot rank features and an independently stable set."""
    sensors = [col for col in test if col.startswith('s') and col[1:].isdigit()]
    common = [c for c in sensors if c in train and
              train[c].notna().any() and test[c].notna().any() and
              train[c].nunique(dropna=True) > 1]
    raw_train = train[common].astype('float32').copy()
    raw_test = test[common].astype('float32').copy()
    rank_train = train[common].rank(pct=True).astype('float32')
    rank_test = test[common].rank(pct=True).astype('float32')
    for a, src in ((raw_train, train), (raw_test, test),
                   (rank_train, train), (rank_test, test)):
        a['missing_count'] = src[common].isna().sum(axis=1).astype('float32')

    # Severe cross-depot drift is especially risky when estimating costs.
    # Use a much more conservative feature subset for positive repair-cost models.
    stable_cols = []
    excluded = []
    for c in common:
        a = train[c].dropna().to_numpy()
        b = test[c].dropna().to_numpy()
        if len(a) > 500 and len(b) > 500 and ks_2samp(a, b, mode='asymp').statistic < 0.15:
            stable_cols.append(c)
        else:
            excluded.append(c)
    stable_train_rank = train[stable_cols].rank(pct=True).fillna(-999).astype('float32')
    stable_test_rank = test[stable_cols].rank(pct=True).fillna(-999).astype('float32')
    # LightGBM respects missingness, unlike CatBoost which needs explicit values.
    tw_train = stable_train_rank.replace(-999, np.nan)
    tw_test = stable_test_rank.replace(-999, np.nan)
    print(f'Sensors usable across depots: {len(common)}')
    print(f'Severely drifted/unstable sensors excluded from cost model: {excluded}')
    return (raw_train, raw_test), (rank_train, rank_test), (stable_train_rank, stable_test_rank), (tw_train, tw_test)




In [10]:
def train_failure_models(raw, ranked, y, costs, folds):
    """Cross-validated failure risk and cost-weighted risk models."""
    weights = np.ones(len(y), dtype=np.float64)
    cp = np.clip(costs[y == 1], 30.0, 2500.0)
    weights[y == 1] = cp / cp.mean()
    predictions = {}
    for family, (x_train, x_test) in [('raw', raw), ('rank', ranked)]:
        for weighted in (False, True):
            oof = np.zeros(len(y), dtype=np.float64)
            test_prediction = np.zeros(len(x_test), dtype=np.float64)
            for fold, (fit, val) in enumerate(folds):
                clf = lgb.LGBMClassifier(
                    objective='binary', n_estimators=700, learning_rate=0.035,
                    num_leaves=15, min_child_samples=65, reg_lambda=8,
                    colsample_bytree=0.85, subsample=0.85, subsample_freq=1,
                    random_state=RANDOM_STATE + fold + 1, n_jobs=N_JOBS,
                    verbosity=-1)
                arguments = dict(
                    eval_set=[(x_train.iloc[val], y[val])],
                    eval_metric='binary_logloss',
                    callbacks=[lgb.early_stopping(70, first_metric_only=True, verbose=False)])
                if weighted:
                    arguments['sample_weight'] = weights[fit]
                clf.fit(x_train.iloc[fit], y[fit], **arguments)
                oof[val] = clf.predict_proba(x_train.iloc[val])[:, 1]
                test_prediction += clf.predict_proba(x_test)[:, 1] / len(folds)
            key = (family, weighted)
            predictions[key] = (oof, test_prediction)
            print(f'Failure {family}/{"weighted" if weighted else "standard"}: '
                  f'ROC-AUC={roc_auc_score(y, oof):.5f}, '
                  f'PR-AUC={average_precision_score(y, oof):.5f}')
    def ensemble(which, weighted):
        return 0.30*predictions[('raw', weighted)][which] + 0.70*predictions[('rank', weighted)][which]
    return ensemble(0,False),ensemble(1,False),ensemble(0,True),ensemble(1,True)




In [11]:
def train_positive_cost_models(cost_features, y, costs, folds):
    """Predict log(1+repair cost) conditional on an actual recorded failure."""
    x_train, x_test = cost_features
    OOF = []; TEST = []
    for depth, ensemble_weight in ((3, 0.40), (5, 0.60)):
        log_cost_oof = np.zeros(len(y), dtype=np.float64)
        log_cost_test = np.zeros(len(x_test), dtype=np.float64)
        for fold, (fit, val) in enumerate(folds):
            positive_fit = fit[y[fit] == 1]
            model = CatBoostRegressor(
                iterations=650, depth=depth, learning_rate=0.045,
                l2_leaf_reg=10, loss_function='RMSE', thread_count=N_JOBS,
                random_seed=RANDOM_STATE + fold, verbose=False,
                allow_writing_files=False)
            model.fit(x_train.iloc[positive_fit], np.log1p(costs[positive_fit]))
            log_cost_oof[val] = model.predict(x_train.iloc[val])
            log_cost_test += model.predict(x_test) / len(folds)
        OOF.append(ensemble_weight * log_cost_oof)
        TEST.append(ensemble_weight * log_cost_test)
        print(f'Positive repair cost depth={depth}: '
              f'out-of-fold log R²={r2_score(np.log1p(costs[y==1]), log_cost_oof[y==1]):.4f}')
    return np.sum(OOF,axis=0), np.sum(TEST,axis=0)


def train_tweedie_cost_model(features, y, costs, folds):
    """Direct expected-loss prediction (Tweedie compound loss)."""
    x_train, x_test = features
    oof = np.zeros(len(y), dtype=np.float64)
    preds = np.zeros(len(x_test), dtype=np.float64)
    for fold, (fit, val) in enumerate(folds):
        model = lgb.LGBMRegressor(
            objective='tweedie', tweedie_variance_power=1.2,
            n_estimators=500, learning_rate=0.03, num_leaves=12,
            min_child_samples=75, reg_lambda=10, colsample_bytree=0.90,
            random_state=RANDOM_STATE + fold, n_jobs=N_JOBS, verbosity=-1)
        model.fit(x_train.iloc[fit], costs[fit],
                  eval_set=[(x_train.iloc[val], costs[val])],
                  eval_metric='tweedie',
                  callbacks=[lgb.early_stopping(65, first_metric_only=True, verbose=False)])
        oof[val] = model.predict(x_train.iloc[val])
        preds += model.predict(x_test) / len(folds)
    return oof, preds




In [12]:
def conditional_expected_cost(log_cost, residuals, max_cost):
    """E[repair_cost | failure, sensors] with nonnegative and upper clipping."""
    A = np.exp(log_cost)
    N = len(residuals)
    prefix = np.r_[0., np.cumsum(residuals)]
    zeros = np.searchsorted(residuals, 1. / A, side='right')
    uncapped = np.searchsorted(residuals, (max_cost + 1) / A, side='right')
    mean = (A * (prefix[uncapped] - prefix[zeros]) - (uncapped - zeros)
            + (N-uncapped) * max_cost) / N
    return np.maximum(mean, 0.), zeros, prefix, A


def decide(p_failure, p_weighted, direct_loss, log_cost, residuals, max_cost):
    """Optimize each vehicle's inspection choice and parts budget.

    If C is positive repair cost and p is failure risk, demand is 0 with
    probability 1-p and C with probability p.
    The optimal quantile for unconditional parts demand is:
       Q = 0.20/(0.20+0.02) = 0.90909.
    Equivalently choose conditional quantile 1-0.02/(0.22*p), or budget 0
    if that expression is nonpositive. Inspections occur only if the
    predicted avoided failure cost exceeds inspection + parts penalties.
    """
    p_budget = np.clip(BUDGET_PROB_SCALE * p_failure, 0., 1.)
    mean_cost, zeros, prefix, A = conditional_expected_cost(log_cost, residuals, max_cost)
    q = 1.0 - SURPLUS_RATE / (SHORTAGE_RATE+SURPLUS_RATE) / np.maximum(p_budget, 1e-12)
    n = len(residuals)
    budgets = np.zeros(len(p_budget), dtype=np.float64)
    active = q > 0
    fractional = np.clip(q[active] * (n-1),0,n-1)
    lower = np.floor(fractional).astype(int)
    upper = np.ceil(fractional).astype(int)
    fraction = fractional - lower
    value = residuals[lower] * (1.-fraction) + residuals[upper]*fraction
    budgets[active] = np.clip(A[active] * value - 1, 0, max_cost)
    budgets = np.rint(budgets)
    count_le = np.searchsorted(residuals, (budgets + 1.) / A, side='right')
    surplus_pos = (budgets*zeros + (budgets+1)*(count_le-zeros)
                   - A*(prefix[count_le]-prefix[zeros])) / n
    surplus_pos = np.maximum(surplus_pos, 0.)
    shortage_pos = np.maximum(mean_cost - budgets + surplus_pos, 0.)
    inspect_loss = (INSPECTION_COST + SHORTAGE_RATE*p_budget*shortage_pos
                    + SURPLUS_RATE*((1.-p_budget)*budgets + p_budget*surplus_pos))
    # Separate severity-aware failure model provides a second loss estimate.
    calibrated_risk = np.clip(INSPECTION_RISK_SCALE*(
        (1.-TWEEDIE_RISK_WEIGHT)*(.5*p_failure+.5*p_weighted)
        + TWEEDIE_RISK_WEIGHT*(direct_loss/np.maximum(mean_cost, 1e-9))),0.,1.)
    skip_loss = calibrated_risk * mean_cost
    flag = (skip_loss > inspect_loss).astype(np.int8)
    budgets = np.where(flag == 1, budgets, 0).astype(np.int64)
    return flag, budgets


def realized_proxy_cost(flag, budget, actual_cost):
    return float(np.sum(
        INSPECTION_COST*flag + flag*(
            SHORTAGE_RATE*np.maximum(actual_cost-budget,0)
            +SURPLUS_RATE*np.maximum(budget-actual_cost,0))
        + (1-flag)*actual_cost))




In [13]:
def main():
    data_dir = locate_data()
    out_dir = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else data_dir
    print('Reading:',data_dir)
    train = pd.read_csv(data_dir/'train.csv')
    test = pd.read_csv(data_dir/'test.csv')
    sample = pd.read_csv(data_dir/'sample_submission.csv')
    y = train['failure'].to_numpy(dtype=np.int8)
    costs = train['repair_cost'].fillna(0).to_numpy(dtype=np.float64)
    assert np.all(costs>=0)
    raw, ranked, positive_features, tweedie_features = preprocess(train,test)
    folds=list(StratifiedKFold(n_splits=N_FOLDS,shuffle=True,random_state=RANDOM_STATE).split(np.zeros(len(y)),y))
    p_oof, p_test, pw_oof, pw_test = train_failure_models(raw,ranked,y,costs,folds)
    log_oof, log_test = train_positive_cost_models(positive_features,y,costs,folds)
    direct_oof, direct_test = train_tweedie_cost_model(tweedie_features,y,costs,folds)
    positive = (y == 1)
    residual = np.sort(np.exp(np.log1p(costs[positive])-log_oof[positive]))
    max_cost = float(costs[positive].max())
    val_flag,val_budget = decide(p_oof,pw_oof,direct_oof,log_oof,residual,max_cost)
    print('Proxy cost for never inspecting:',round(costs.sum(),2))
    print('Out-of-fold proxy cost:',round(realized_proxy_cost(val_flag,val_budget,costs),2))
    print('Old policy proxy cost was approximately 55277; evaluate relative to that baseline.')
    print('Out-of-fold inspections:',int(val_flag.sum()))
    flag,budget = decide(p_test,pw_test,direct_test,log_test,residual,max_cost)
    preds = pd.DataFrame({'id':test['id'],'flag':flag,'budget':budget})
    submission = sample[['id']].merge(preds,on='id',how='left',validate='one_to_one')
    assert len(submission)==len(test)==len(sample)
    assert submission['flag'].notna().all() and submission['budget'].notna().all()
    assert set(submission['flag'].unique()).issubset({0,1})
    assert (submission['budget']>=0).all()
    assert (submission.loc[submission['flag']==0,'budget']==0).all()
    outfile = out_dir/'submission.csv'
    submission[['id','flag','budget']].to_csv(outfile,index=False)
    print('Saved:',outfile)
    print('Number of vehicles:',len(submission))
    print('Vehicles marked for inspection:',int(submission['flag'].sum()))
    print('Nonzero spare-parts budgets:',int((submission['budget']>0).sum()))
    print('Total allocated parts budget:',int(submission['budget'].sum()))
    print(submission.head().to_string(index=False))


main()


Reading: /kaggle/input/competitions/mlprojectfleet2
Sensors usable across depots: 168
Severely drifted/unstable sensors excluded from cost model: ['s054', 's055', 's056', 's057', 's059', 's060', 's061', 's062', 's063', 's120', 's122']
Failure raw/standard: ROC-AUC=0.98791, PR-AUC=0.52350
Failure raw/weighted: ROC-AUC=0.98655, PR-AUC=0.50489
Failure rank/standard: ROC-AUC=0.98739, PR-AUC=0.52731
Failure rank/weighted: ROC-AUC=0.98648, PR-AUC=0.50066
Positive repair cost depth=3: out-of-fold log R²=0.6433
Positive repair cost depth=5: out-of-fold log R²=0.6545
Proxy cost for never inspecting: 195476.0
Out-of-fold proxy cost: 47609.9
Old policy proxy cost was approximately 55277; evaluate relative to that baseline.
Out-of-fold inspections: 1410
Saved: /kaggle/working/submission_v2.csv
Number of vehicles: 20000
Vehicles marked for inspection: 836
Nonzero spare-parts budgets: 620
Total allocated parts budget: 297363
    id  flag  budget
t00001     0       0
t00002     0       0
t00003     0